# RVU ID card detector: train the wearing / not wearing model

Runs on a free Colab T4 GPU in roughly an hour (two model sizes are trained). Use **Runtime > Run all**.

Data: Roboflow Universe `id-card-detection-ozsif/id-detectionn-2` version 1 (CC BY 4.0), 2,720 images, classes `id_not_wearing` and `id_wearing`.

Output: `rvu_idcard_models.zip` with both models as ONNX, their test scores and training curves.

In [ ]:
!nvidia-smi -L || echo 'NO GPU: set Runtime > Change runtime type > T4 GPU'
!pip -q install ultralytics onnx onnxslim onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Where the dataset zip comes from. Leave DATA_URL empty to be asked for a Roboflow API key.
DATA_URL = ""
import json, os, urllib.request
if not DATA_URL:
    from getpass import getpass
    key = getpass('Roboflow API key: ')
    api = 'https://api.roboflow.com/id-card-detection-ozsif/id-detectionn-2/1/yolov8?api_key=' + key
    DATA_URL = json.load(urllib.request.urlopen(api))['export']['link']
os.makedirs('/content/data', exist_ok=True)
urllib.request.urlretrieve(DATA_URL, '/content/data/ds.zip')
!cd /content/data && unzip -q -o ds.zip && rm ds.zip
open('/content/data/data.yaml', 'w').write(
    'path: /content/data\ntrain: train/images\nval: valid/images\ntest: test/images\n'
    'names:\n  0: id_not_wearing\n  1: id_wearing\n')
for split in ('train', 'valid', 'test'):
    print(split, len(os.listdir(f'/content/data/{split}/images')), 'images')

In [ ]:
from ultralytics import YOLO
# Two sizes: 'n' is the fast one for laptops without a GPU, 's' is slower but more accurate.
# Both are trained so the choice is made from measured numbers, not a guess.
RUNS = {'n': 'yolo11n.pt', 's': 'yolo11s.pt'}
for tag, weights in RUNS.items():
    YOLO(weights).train(data='/content/data/data.yaml', epochs=80, patience=20, imgsz=640,
                        batch=32, cos_lr=True, close_mosaic=10, degrees=5, fliplr=0.5,
                        project='/content/runs', name='idcard_' + tag, exist_ok=True, plots=True)
    print('TRAINED', tag)

In [ ]:
import json, shutil
os.makedirs('/content/out', exist_ok=True)
summary = {}
for tag in RUNS:
    run = f'/content/runs/idcard_{tag}'
    best = YOLO(run + '/weights/best.pt')
    m = best.val(data='/content/data/data.yaml', split='test', project='/content/runs', name='test_' + tag, exist_ok=True)
    summary[tag] = {'map50': float(m.box.map50), 'map50_95': float(m.box.map),
                    'per_class': {best.names[i]: {'precision': float(m.box.p[k]), 'recall': float(m.box.r[k])}
                                  for k, i in enumerate(m.box.ap_class_index)}}
    print('RESULT', tag, json.dumps(summary[tag]))
    onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
    shutil.copy(onnx_path, f'/content/out/idcard_{tag}.onnx')
    shutil.copy(run + '/weights/best.pt', f'/content/out/idcard_{tag}.pt')
    for extra in ('results.csv', 'results.png', 'confusion_matrix.png'):
        if os.path.exists(f'{run}/{extra}'):
            shutil.copy(f'{run}/{extra}', f'/content/out/{tag}_{extra}')
json.dump(summary, open('/content/out/summary.json', 'w'), indent=1)

In [ ]:
shutil.make_archive('/content/rvu_idcard_models', 'zip', '/content/out')
print('ALL DONE', os.path.getsize('/content/rvu_idcard_models.zip') // 1024, 'KB')
from google.colab import files
files.download('/content/rvu_idcard_models.zip')